# 13 -- SHAP Explainability
**Purpose:** Generate SHAP values for the champion model. Produce top-5 feature explanations per transaction.

**Input:** Champion model + X_test
**Output:** `data/output/shap_explanations.parquet` | SHAP plots in reports/

## Learning Objectives
- Understand what a SHAP value means for a single prediction
- Use TreeExplainer (exact, fast for XGBoost/LightGBM)
- Generate waterfall plots for individual transactions
- Produce summary plots for global feature importance
- Understand the difference between SHAP and feature importance

## Business Context
CrossBorderPay must explain every RED-tier decision to the customer's relationship manager.
"Your transaction was blocked" is not acceptable. "Your transaction was blocked because:
amount is 4.2x your 90-day average, beneficiary country is FATF high-risk, and transaction
was submitted at 2:47 AM" is a defensible decision under PMLA audit.

## Output Format (PoC Plan V5 -- Section 9)
For each transaction: top 5 features by absolute SHAP contribution.
Format: Feature Name | Feature Value | Contribution Score
NO plain-language narratives. NO reason codes. Data only.

## Step 1 -- Generate SHAP Values
Use shap.TreeExplainer(champion_model). Compute SHAP values on X_test.
Print shape of shap_values array. What does each row represent? What does each column represent?

## Step 2 -- Global Summary Plot
Plot shap.summary_plot(). Which feature has the highest mean absolute SHAP value?
Does this match your feature importance from Notebook 08/09?

## Step 3 -- Individual Transaction Explanation
Pick one RED-tier fraud transaction from the test set. Plot shap.waterfall_plot() for it.
Interpret: which feature pushed the score highest? Does the explanation make sense given the transaction?

## Step 4 -- Build Explanation Output
For every test transaction: extract top 5 SHAP features (name, value, contribution).
Save to data/output/shap_explanations.parquet in the format from Section 9 of PoC Plan V5.

## Definition of Done
- [ ] SHAP values generated using TreeExplainer
- [ ] Global summary plot produced
- [ ] At least 3 individual transaction waterfall plots -- one fraud, one legitimate, one borderline
- [ ] Top-5 SHAP explanation table built for all test transactions
- [ ] Output saved to data/output/shap_explanations.parquet
- [ ] You can explain what a negative SHAP value means

In [1]:
import pandas as pd
import shap
from xgboost import XGBClassifier

X_train = pd.read_parquet(r"../data/processed\X_train.parquet")
X_test  = pd.read_parquet(r"../data/processed\X_test.parquet")
y_train = pd.read_parquet(r"../data/processed\y_train.parquet").squeeze()
y_test  = pd.read_parquet(r"../data/processed\y_test.parquet").squeeze()

model = XGBClassifier(
    max_depth=5,
    n_estimators=400,
    scale_pos_weight=202,
    learning_rate=0.1,
    random_state=42,
    eval_metric="aucpr",
    verbosity=0
)
model.fit(X_train, y_train)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

print(f"SHAP values shape: {shap_values.shape}")


SHAP values shape: (10160, 46)


In [2]:
import numpy as np

feature_names = X_test.columns.tolist()

mean_abs_shap = np.abs(shap_values).mean(axis=0)

importance_df = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": mean_abs_shap
}).sort_values("mean_abs_shap", ascending=False)

print("Top 10 most important features:")
print(importance_df.head(10).to_string(index=False))


Top 10 most important features:
                     feature  mean_abs_shap
          invoice_match_flag       3.099853
              log_amount_usd       2.430843
       turnover_to_txn_ratio       1.027665
           amount_zscore_90d       1.008134
          is_primary_account       0.945826
   time_since_last_login_hrs       0.834095
         days_since_last_txn       0.613755
days_since_account_last_used       0.591624
             submission_hour       0.544270
             is_round_number       0.469938


In [3]:
import pandas as pd

y_test_reset = y_test.reset_index(drop=True)
fraud_indices = y_test_reset[y_test_reset == 1].index.tolist()

# Pick the first fraud transaction
idx = fraud_indices[0]

single_shap = pd.DataFrame({
    "feature": feature_names,
    "shap_value": shap_values[idx],
    "feature_value": X_test.iloc[idx].values
}).sort_values("shap_value", ascending=False)

print(f"Transaction index: {idx}")
print("\nTop 5 features pushing TOWARD fraud:")
print(single_shap.head(5).to_string(index=False))
print("\nTop 5 features pushing TOWARD legit:")
print(single_shap.tail(5).to_string(index=False))


Transaction index: 146

Top 5 features pushing TOWARD fraud:
           feature  shap_value  feature_value
 amount_zscore_90d    6.938835         4.6339
invoice_match_flag    2.630155         0.0000
is_new_beneficiary    1.318738         1.0000
    is_new_country    0.276252         1.0000
       has_invoice    0.269410         1.0000

Top 5 features pushing TOWARD legit:
              feature  shap_value  feature_value
  corridor_fatf_score   -0.458741         3.0000
turnover_to_txn_ratio   -0.528646         0.1226
  days_since_last_txn   -0.560334         1.5100
         biz_Exporter   -0.686866         1.0000
 failed_txn_count_30d   -0.739401         3.0000


In [4]:
y_prob = model.predict_proba(X_test)[:, 1]

idx = fraud_indices[0]
prob = y_prob[idx]
verdict = "FRAUD" if prob >= 0.9217 else "LEGIT"

single_shap = pd.DataFrame({
    "feature": feature_names,
    "shap_value": shap_values[idx],
    "feature_value": X_test.iloc[idx].values
}).sort_values("shap_value", ascending=False)

print(f"Transaction index: {idx}")
print(f"Fraud probability: {prob:.4f} → {verdict} (threshold: 0.9217)")
print("\nTop 5 features pushing TOWARD fraud:")
print(single_shap.head(5).to_string(index=False))
print("\nTop 5 features pushing TOWARD legit:")
print(single_shap.tail(5).to_string(index=False))


Transaction index: 146
Fraud probability: 0.9988 → FRAUD (threshold: 0.9217)

Top 5 features pushing TOWARD fraud:
           feature  shap_value  feature_value
 amount_zscore_90d    6.938835         4.6339
invoice_match_flag    2.630155         0.0000
is_new_beneficiary    1.318738         1.0000
    is_new_country    0.276252         1.0000
       has_invoice    0.269410         1.0000

Top 5 features pushing TOWARD legit:
              feature  shap_value  feature_value
  corridor_fatf_score   -0.458741         3.0000
turnover_to_txn_ratio   -0.528646         0.1226
  days_since_last_txn   -0.560334         1.5100
         biz_Exporter   -0.686866         1.0000
 failed_txn_count_30d   -0.739401         3.0000


In [5]:
y_prob_legit = model.predict_proba(X_test)[:, 0]
y_prob_fraud = model.predict_proba(X_test)[:, 1]

idx = fraud_indices[0]
prob_fraud = y_prob_fraud[idx]
prob_legit = y_prob_legit[idx]
verdict = "FRAUD" if prob_fraud >= 0.9217 else "LEGIT"

single_shap = pd.DataFrame({
    "feature": feature_names,
    "shap_value": shap_values[idx],
    "feature_value": X_test.iloc[idx].values
}).sort_values("shap_value", ascending=False)

print(f"Transaction index:   {idx}")
print(f"Fraud probability:   {prob_fraud:.4f}")
print(f"Legit probability:   {prob_legit:.4f}")
print(f"Verdict:             {verdict} (threshold: 0.9217)")
print("\nTop 5 features pushing TOWARD fraud:")
print(single_shap.head(5).to_string(index=False))
print("\nTop 5 features pushing TOWARD legit:")
print(single_shap.tail(5).to_string(index=False))


Transaction index:   146
Fraud probability:   0.9988
Legit probability:   0.0012
Verdict:             FRAUD (threshold: 0.9217)

Top 5 features pushing TOWARD fraud:
           feature  shap_value  feature_value
 amount_zscore_90d    6.938835         4.6339
invoice_match_flag    2.630155         0.0000
is_new_beneficiary    1.318738         1.0000
    is_new_country    0.276252         1.0000
       has_invoice    0.269410         1.0000

Top 5 features pushing TOWARD legit:
              feature  shap_value  feature_value
  corridor_fatf_score   -0.458741         3.0000
turnover_to_txn_ratio   -0.528646         0.1226
  days_since_last_txn   -0.560334         1.5100
         biz_Exporter   -0.686866         1.0000
 failed_txn_count_30d   -0.739401         3.0000


In [6]:
import joblib
joblib.dump(model, r"../data/processed\xgboost_champion.pkl")
print("Champion model saved — XGBoost 2.1.3 + SHAP compatible.")


Champion model saved — XGBoost 2.1.3 + SHAP compatible.
